In [70]:
import pandas as pd

In [6]:
from pathlib import Path

files = Path("../../data").glob("*.csv")

for file in files:
    df = pd.read_csv(file)
    print(f"{file} {df.columns}")



../../data/humanitarian-response-plans.csv Index(['code', 'internalId', 'startDate', 'endDate', 'planVersion',
       'categories', 'locations', 'years', 'origRequirements',
       'revisedRequirements'],
      dtype='str')
../../data/ProjectSummaryWithLocationAndCluster20260418111139715.csv Index(['PooledFundId', 'PooledFundName', 'ChfId', 'ChfProjectCode',
       'ExternalProjectCode', 'AllocationType', 'AllocationYear',
       'AllocationSourceID', 'AllocationSourceName', 'OrganizationName',
       'OrganizationType', 'ProjectTitle', 'ProjectDuration', 'Budget',
       'EnvironmentMarker', 'GenderMarker', 'ActualStartDate', 'ActualEndDate',
       'ProjectStatus', 'ProjectStatusId', 'ProjectStatusCode',
       'ProcessStatus', 'ProcessStatusId', 'PartnerCode', 'Cluster',
       'ClusterPercentage', 'AdminLocation1TypeName', 'AdminLocation1',
       'AdminLocation1Latitude', 'AdminLocation1Longitude', 'AdminLevel1PCode',
       'AdminLevel1Percentage'],
      dtype='str')
../../data/

/var/folders/kn/fg_k6n0119d337lwnvhwnmnw0000gn/T/ipykernel_16502/446345361.py:6: DtypeWarning: Columns (0: Admin 1 PCode, 1: Admin 1 Name, 2: Admin 3 PCode, 3: Admin 3 Name, 4: Population, 5: In Need, 6: Targeted, 7: Affected, 8: Reached, 9: Info) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


../../data/hpc_hno_2025.csv Index(['Country ISO3', 'Admin 1 PCode', 'Admin 1 Name', 'Admin 2 PCode',
       'Admin 2 Name', 'Admin 3 PCode', 'Admin 3 Name', 'Description',
       'Cluster', 'Category', 'Population', 'In Need', 'Targeted', 'Affected',
       'Reached', 'Info'],
      dtype='str')


/var/folders/kn/fg_k6n0119d337lwnvhwnmnw0000gn/T/ipykernel_16502/446345361.py:6: DtypeWarning: Columns (0: Admin 1 PCode, 1: Admin 1 Name, 2: Admin 2 PCode, 3: Admin 2 Name, 4: Admin 3 PCode, 5: Admin 3 Name, 6: Population, 7: In Need, 8: Targeted, 9: Affected, 10: Reached, 11: Info) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


../../data/hpc_hno_2024.csv Index(['Country ISO3', 'Admin 1 PCode', 'Admin 1 Name', 'Admin 2 PCode',
       'Admin 2 Name', 'Admin 3 PCode', 'Admin 3 Name', 'Description',
       'Cluster', 'Category', 'Population', 'In Need', 'Targeted', 'Affected',
       'Reached', 'Info'],
      dtype='str')
../../data/cod_population_admin2.csv Index(['ISO3', 'Country', 'ADM1_PCODE', 'ADM1_NAME', 'ADM2_PCODE', 'ADM2_NAME',
       'ADM3_PCODE', 'ADM3_NAME', 'ADM4_PCODE', 'ADM4_NAME',
       'Population_group', 'Gender', 'Age_range', 'Age_min', 'Age_max',
       'Population', 'Reference_year', 'Source', 'Contributor'],
      dtype='str')
../../data/cod_population_admin3.csv Index(['ISO3', 'Country', 'ADM1_PCODE', 'ADM1_NAME', 'ADM2_PCODE', 'ADM2_NAME',
       'ADM3_PCODE', 'ADM3_NAME', 'ADM4_PCODE', 'ADM4_NAME',
       'Population_group', 'Gender', 'Age_range', 'Age_min', 'Age_max',
       'Population', 'Reference_year', 'Source', 'Contributor'],
      dtype='str')
../../data/fts_incoming_funding_glo

In [72]:
import pandas as pd

def aggregate_needs(year):
    pd.options.display.float_format = '{:,.0f}'.format
    df_needs = pd.read_csv(f"../../data/hpc_hno_{year}.csv", low_memory=False)
    df_needs = df_needs[1:] 

    df_needs["In Need"] = df_needs["In Need"].astype(str).str.replace(',', '')
    df_needs["In Need"] = pd.to_numeric(df_needs["In Need"], errors='coerce').fillna(0)

    df_clean = df_needs[df_needs['Admin 1 PCode'].isna() & df_needs['Category'].isna()]

    df_pivot = pd.pivot_table(
        df_clean, 
        values='In Need', 
        index='Country ISO3', 
        columns='Cluster', 
        aggfunc='sum', 
        fill_value=0
    ).reset_index()

    cluster_cols = [col for col in df_pivot.columns if col != 'Country ISO3']
    df_pivot = df_pivot.rename(columns={col: f"In Need - {col}" for col in cluster_cols})

    return df_pivot

df_needs2025 = aggregate_needs("2025")
df_needs2026 = aggregate_needs("2024")

In [66]:
# import pandas as pd

def add_country_population_info(df_pivot):
    df_pop = pd.read_csv("../../data/cod_population_admin0.csv", low_memory=False)
    df_pop["Population"] = df_pop["Population"].astype(str).str.replace(',', '')
    df_pop["Population"] = pd.to_numeric(df_pop["Population"], errors='coerce').fillna(0)
    df_pop_totals = df_pop[df_pop['Gender'].str.lower().isin(['all', 'total', 'both']) | df_pop['Gender'].isna()]
    df_pop_clean = df_pop_totals.groupby('ISO3')['Population'].max().reset_index()
    df_pop_clean = df_pop_clean.rename(columns={'ISO3': 'Country ISO3', 'Population': 'Total National Population'})
    df_master = pd.merge(df_pivot, df_pop_clean, on='Country ISO3', how='inner')
    return df_master

df_2025_popinfo = add_country_population_info(df_needs2025)
df_2025_popinfo


# df_2025['Relative In Need - EDU [%]'] = (df_2025['In Need - EDU'] / df_2025['Total National Population']) * 100
# df_2025


,Country ISO3,In Need - ALL,In Need - CCM,In Need - CSS,In Need - EDU,In Need - ERY,In Need - FSC,In Need - HEA,In Need - LOG,In Need - MPC,...,In Need - NUT,In Need - PRO,In Need - PRO-CPN,In Need - PRO-GBV,In Need - PRO-HLP,In Need - PRO-MIN,In Need - SHL,In Need - TEL,In Need - WSH,Total National Population
0,AFG,"22,887,726",0,0,"8,887,445",0,"14,764,729","14,280,440",0,0,...,"7,801,981","36,472,072","9,352,775","14,173,291","3,392,653","4,369,984","5,757,529",0,"20,973,271",40411869
1,BFA,"5,915,136","1,348,415",0,"1,992,887",0,"2,800,731","2,031,499",0,0,...,"1,314,600","3,756,956","1,842,924","1,322,131","1,573,606","1,909,220","2,490,889",0,"2,830,634",23558491
2,CMR,"3,322,058",0,0,"1,496,550",0,"2,483,832","1,511,471",0,0,...,"562,839","3,594,609","1,198,337","1,147,218","840,995",0,"1,431,605",0,"2,155,114",29442318
3,COD,"21,241,767","1,286,038",0,"1,868,832",0,"17,611,104","12,935,678",0,0,...,"6,472,173","12,870,488","4,006,648","6,213,757","2,597,302","2,419,968","4,035,452",0,"6,216,647",103199358
4,COL,"9,053,352",0,0,"3,003,411","4,288,499","8,180,565","6,013,741",0,0,...,0,"5,477,222","2,160,165","3,281,409",0,"688,477","3,975,631",0,"4,260,446",53216592
5,GTM,"2,163,953","185,203",0,"599,290",0,"2,604,233","673,781",0,0,...,"2,200,000","1,708,044","398,015","1,166,612",0,0,0,0,"762,174",17843132
6,HND,"1,640,224","513,000",0,"396,425",0,"1,224,447","1,109,775",0,0,...,"434,415","1,169,056","524,179","865,234",0,0,0,0,"1,165,525",9892674
7,HTI,"5,986,456","1,054,487",0,"1,474,332",0,"5,535,548","4,214,625",0,0,...,"1,170,492","2,976,595","1,611,291","1,535,819",0,0,"3,645,045",0,"3,854,002",11899555
8,MLI,"6,431,534",0,0,"1,810,162",0,"2,908,655","3,742,972",0,0,...,"2,838,298","4,890,568","1,728,161","2,271,395","1,067,287","1,344,844","2,335,122",0,"3,256,054",17907114
9,MOZ,"1,305,029","446,611",0,"278,279",0,"1,113,323","810,850",0,0,...,"268,485","1,726,207","890,107","533,025","224,429",0,"774,717",0,"1,211,906",35163992


In [69]:
import numpy as np


def add_relative_inneed(df_master):
    in_need_cols = [col for col in df_master.columns if col.startswith('In Need - ')]
    for col in in_need_cols:
        sector_name = col.replace('In Need - ', '')
        new_col_name = f"% Pop In Need - {sector_name}"
        df_master[new_col_name] = (df_master[col] / df_master['Total National Population']) * 100
        df_master[new_col_name] = df_master[new_col_name].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df_master

df_2025_rel = add_relative_inneed(df_2025_popinfo)
df_2025_rel

,Country ISO3,In Need - ALL,In Need - CCM,In Need - CSS,In Need - EDU,In Need - ERY,In Need - FSC,In Need - HEA,In Need - LOG,In Need - MPC,...,% Pop In Need - MS,% Pop In Need - NUT,% Pop In Need - PRO,% Pop In Need - PRO-CPN,% Pop In Need - PRO-GBV,% Pop In Need - PRO-HLP,% Pop In Need - PRO-MIN,% Pop In Need - SHL,% Pop In Need - TEL,% Pop In Need - WSH
0,AFG,"22,887,726",0,0,"8,887,445",0,"14,764,729","14,280,440",0,0,...,0,19,90,23,35,8,11,14,0,52
1,BFA,"5,915,136","1,348,415",0,"1,992,887",0,"2,800,731","2,031,499",0,0,...,0,6,16,8,6,7,8,11,0,12
2,CMR,"3,322,058",0,0,"1,496,550",0,"2,483,832","1,511,471",0,0,...,2,2,12,4,4,3,0,5,0,7
3,COD,"21,241,767","1,286,038",0,"1,868,832",0,"17,611,104","12,935,678",0,0,...,1,6,12,4,6,3,2,4,0,6
4,COL,"9,053,352",0,0,"3,003,411","4,288,499","8,180,565","6,013,741",0,0,...,0,0,10,4,6,0,1,7,0,8
5,GTM,"2,163,953","185,203",0,"599,290",0,"2,604,233","673,781",0,0,...,0,12,10,2,7,0,0,0,0,4
6,HND,"1,640,224","513,000",0,"396,425",0,"1,224,447","1,109,775",0,0,...,0,4,12,5,9,0,0,0,0,12
7,HTI,"5,986,456","1,054,487",0,"1,474,332",0,"5,535,548","4,214,625",0,0,...,0,10,25,14,13,0,0,31,0,32
8,MLI,"6,431,534",0,0,"1,810,162",0,"2,908,655","3,742,972",0,0,...,2,16,27,10,13,6,8,13,0,18
9,MOZ,"1,305,029","446,611",0,"278,279",0,"1,113,323","810,850",0,0,...,0,1,5,3,2,1,0,2,0,3
